#Covered Today:

1. Deep Dive into Transformers, Quantization, and Neural Networks
2. Working with Hugging Face Transformers Low-Level API and Quantization
3. Inside LLaMA: PyTorch Model Architecture and Token Embeddings
4. Inside LLaMA: Decoder Layers, Attention, and Why Non-Linearity Matters
5. Running Open Source LLMs: Phi, Gemma, Qwen & DeepSeek with Hugging Face

In [1]:
!pip install -q --upgrade bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.0 MB/s eta 0:00:00


In [2]:
# like always we first login to huggingface
from huggingface_hub import login
from google.colab import userdata


hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [3]:
# now we check if GPU is connected or not
gpu_info = !nvidia-smi
gpu_info = "\n".join(gpu_info)

if gpu_info.find("Tesla T4") >= 1 and gpu_info.find("CUDA Version") >= 1:
  print("Nvidia T4 GPU connected")
  print(gpu_info)
else:
  print("Not Connected to GPU, please check")

Nvidia T4 GPU connected
Sun Aug 30 04:00:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------

We start by moving towards quantization. Most modern LLMs have billions of parameters and these parameters are stored in 16 bits or 2 bytes. Now, when billions of such paramters are concerned, this 2 bytes becomes a huge number. For example, having a model with 1 billion parameters, with 2 bytes per parameter, we are looking at 2GB of space only for storing the parameters. To run this model, we would need more overhead, such as CUDA Driver Baseline and KV Cache & Activations. This will take the memory needed to run one such model to around 3 GBs.

Now, this example was just used to share the numbers, and in real life, a 1B model is considered very small, also SLM or small language models.

Our usual models are 8B in size or more. Now, for storing only, a 8B model would need 16 gigs of RAM, plus some overhead to run it(inference), and that number would be around 18 to 20GB. And naturally, bigger the size of the model, bigger hardware is needed to run it. This is where quantization comes into the picture.

Quantization means converting these parameters which are usually stored in 2bytes/16 bits to smaller sizes or converting high-precision decimal values into lower-precision integers or smaller floats to save memory. Basically in layman language we can understand it like this: we convert the model's parameters(which are numbers using 16 bits) in such a way, that we reduce the size from 16 bits to 8 bits or 4 bits depending upon how much quantization we are doing.

# Let us understand this in numbers:
- We have one 8B model:, this model when not quantized, would be using 16 gigs of ram just to store it's weights and more headroom such as 3 to 4 gigs to run it.
- Now, if we reduce the model size by half or quantize it to 8 bits, the weight size would become 8 gigs and another 2 to 3 gigs would be needed to run it. That makes it 10 to 11 GBs.
- Further, if we reduce it to 4 bits, then that same 8B parameter model, would need 4 GBs for storage and 2 to 3 GBs to run it, making it runnable with 6 to 7 GBs of memory.

Another point to be noted here is that there is a very minute loss of accuracy or reasoning when we do quantization, however, the smaller models feel more pinch due to lack of overlapping parameters. While models of large sizes such as 70B or more, use multiple pathways to store same information or learn the same fact, therefore if we quantize a model, larger models have almost no impact in terms of accuracy or reasoning.


In [4]:
# lets now see how we can run quantization using tranformers library
# We use a class similar to AutoTokenizer under transformers for quant also, known as BitsAndBytesConfig
from transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM, TextStreamer
# plus, we also need to import pytorch, explanation is further ahead.
import torch

# We talk about AutoModelForCausalLM and TextStreamer later

In [5]:
# this library allows us to define how we want to quantify a model, below is the code:

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

# now, we are storing the quantization configuration in quant_config variable, by using the BitsAndBytesConfig class. Let's have a look at what else we have done inside this class
# 1. load_in_4bit: tells the transformers and bits&bytes class to compress the weights of the model from 16 bits to 4 bits, this is the main task under quantization, which is reducing the model weight size.
# 2. bnb_4bit_use_double_quant: quantizes the scale factors also(scale factors are the scale reduction number for a block of quantized parameters)
# 3. nb_4bit_compute_dtype=torch.bfloat16: a GPU can not work with quantized parameters, so we tell it, that these parameters are to be converted to a 16 bit number, these are stored temporarily and then discarded, and in this command, we are simply telling it, which number format is needed to run this model
# 4. bnb_4bit_quant_type="nf4": Sets the quantization format to NormalFloat 4 (NF4), which is a data type used to store quantized parameters

In [6]:
# we will now be using "meta-llama/Llama-3.2-1B-Instruct"
LLAMA = "meta-llama/Llama-3.2-1B-Instruct"

messages = [
    {"role": "user", "content": "Tell a light hearted joke on a software engineer."}
]

# we start by creating its tokenizer, code to be explained later
tokenizer = AutoTokenizer.from_pretrained(LLAMA)

# then this is a convention, we write this, since tensor needs to be of same dimension, and we add eos tokens to pad tokens
tokenizer.pad_token = tokenizer.eos_token

# then we are applying chat template, here, return_tensor is the parameter which informed the tokenizer to return the token IDs in a specific format. There are multiple formats, such as pt for pytorch, tf for tensor flow etc, but pytorch is most used and other formats or libraries needed are mostly in legacy code. So, by default we use pytorch, hence for return_tensors we use pt. Then finally, when the return tensor outputs the tensor, this is done in the CPU. Once, it is done, .to("cuda") moves it from cpu to gpu.
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
# now till here, we have first created a messages, this is the text that is written by human in NL and now must be transformed in a format that LLM can consume. So, we first create a tokenizer relevant for that LLM. Then, we create the input, by applying chat template and then converting this messages into pytorch tensor format, which gets generated in the CPU and finally we pass it to the GPU. Let's see, how this input looks like below:
print(inputs)


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

{'input_ids': tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,    966,   5033,    220,   2366,     21,    271, 128009, 128006,
            882, 128007,    271,  41551,    264,   3177,   4851,    291,  22380,
            389,    264,   3241,  24490,     13, 128009]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:0')}


In [7]:
# now the model
# here, we first use AutoModelForCausalLM: which automatically figures out the internals of the model being passed in arguments and CausalLM stands for generative LLM(which predicts the next token) or auto build a next token predictor from pretrained weights of LLAMA models, where auto decide on the device(GPU or CPU), with priority being the GPU. We use auto because device_map(from accelerate library) is a smart tool and is great in managing memory use. For small models, everything goes on the GPU and in case of larger ones, any spill over is sent across to the CPU. Finally, we are also passing in the quantization details using the config we created earlier.

model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [8]:
# now the model is loaded in the GPU. Now by simply running model, we can print the internal of the model(possible on pytorch), but we skip the understanding for now.
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm

In [9]:
# lets get the size of the model in the memory, since it will be in bytes, we will get 1 billion bytes, divide it by 10 to power 6 or 1e6, we should get MBs
model.get_memory_footprint() / 1e6
# the model is 1000 mbs or 1GB and while the model is of around 2.5 GBs, it has reduced to 1 GB since, other than parameters the model contains other infomation also, which has not been quantized. But still more than half is also great.

1012.011264

In [10]:
# Now we run the model.
outputs = model.generate(**inputs, max_new_tokens=200)
outputs[0]

tensor([128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
            25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
           220,    966,   5033,    220,   2366,     21,    271, 128009, 128006,
           882, 128007,    271,  41551,    264,   3177,   4851,    291,  22380,
           389,    264,   3241,  24490,     13, 128009, 128006,  78191, 128007,
           271,     32,   3241,  24490,  23291,   1139,    264,   3703,    323,
         10373,    264,  13179,     13,   1666,    568,    596,    274,   5772,
           813,   7172,     11,    568,  53159,    264,   7899,   2019,     11,
           330,  46078,  18623,   9135,   1283,   5992,   2212,     11,    719,
          1070,    596,  19093,  14373,    889,   1436,    617,   1071,    433,
            13,    362,   2478,   4520,   3010,     11,    568,  53159,     11,
           330,  47618,  15845,   9135,  14077,     11,    568,   5992,   2212,
            11,    719,    568,    649, 

In [11]:
tokenizer.decode(outputs[0])

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 30 Aug 2026\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nTell a light hearted joke on a software engineer.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nA software engineer walks into a bar and orders a beer. As he\'s sipping his drink, he hears a voice say, "Nice tie!" He looks around, but there\'s nobody nearby who could have said it. A few minutes later, he hears, "Beautiful shirt!" Again, he looks around, but he can\'t find anyone who might have spoken.\n\nA few more minutes pass, and he hears, "Great haircut!" This time, he decides to investigate. He asks the bartender, "Did you hear those voices?"\n\nThe bartender replies, "Oh, that\'s just the peanuts. They\'re complimentary."<|eot_id|>'

In [12]:
# here, when I used return_dict=False while applying chat template, I got a clean tensor, else I was getting a dict, which was not running in generate as there was a format mismatch, generate expects tensor or tensors, cleanly. If we do not use return_dict=False, then we need to use **inputs. This has been introduced in V5 of transformers.
# Now we clean the memory using code, instead of using GUI:
import gc
del model, inputs, tokenizer, outputs
gc.collect()
torch.cuda.empty_cache()

In [13]:
# now next up, we write one big function to bring it all in one place, before that, let's revise the steps of what is to be done now.
# first the libraries needed: we need torch, from transformers we need bitsandbytes, autotokenizer, automodelcasuallm, accelerate, we also need to install the bitsandbytes version, and streamer also which we shall use next
# so these are
# !pip install -q --upgrade bitsandbytes
# from transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCasualLM, TextStreamer
# we shall try to do this all standalone also at the end of day 5.

# Now, once all the imports are done, we create a tokenizer for the model we want to use, we create messages to be passed on, we also create a quant_config if needed. we then make tokenizer.pad_token=tokenizer.eos_token, then we create the inputs, where we apply chat template to our messages, we also add tokenize=True, we mark the return_tensor='pt' for pytorch, we can also pass in add_generative_prompt, we can also specify if any special tokens are to be added or not, once done, we pass this to cuda. This moves our tensor format input in the GPU.
# Next we have to upload the model to the GPU, for this, we use AutoModelForCasualLM.from_pretrained('model_name', quant_config and auto_device), where quant_config tells if the model is to be quantized and auto_device manages the model in the memory by smartly allocating these between GPU and CPU(in case of any overflow). Now, both the model and the input are in the memory.
# Next up, we call on the model to generate text based on the input, and also pass in total new tokens to be generated. The output is in tensor format, we then encode it using our tokenizer.
# let's first do this, without the bigger function, we do this on our own.
MODEL = 'meta-llama/Llama-3.2-3B-Instruct'
new_tokenizer = AutoTokenizer.from_pretrained(MODEL)

new_quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

messages = [
    {"role": "user", "content": "Tell us a snarky joke about an applied LLM engineer"}
]

new_tokenizer.pad_token = new_tokenizer.eos_token

inputs = new_tokenizer.apply_chat_template(messages, return_tensors='pt', tokenize=True, add_generation_prompt=True).to("cuda")
print(inputs)

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

{'input_ids': tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,    966,   5033,    220,   2366,     21,    271, 128009, 128006,
            882, 128007,    271,  41551,    603,    264,   4224,    847,     88,
          22380,    922,    459,   9435,    445,  11237,  24490, 128009, 128006,
          78191, 128007,    271]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:0')}


In [14]:
model = AutoModelForCausalLM.from_pretrained(MODEL, device_map="auto", quantization_config=new_quant_config)

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [15]:
model.get_memory_footprint() / 1e6

2197.641728

In [16]:
# we can see here, that we downloaded a model of almost 6.5 GBs and after quant, it is 2 GBs.
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNo

In [17]:
outputs = model.generate(**inputs, max_new_tokens=200)

In [18]:
print(outputs[0])

tensor([128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
            25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
           220,    966,   5033,    220,   2366,     21,    271, 128009, 128006,
           882, 128007,    271,  41551,    603,    264,   4224,    847,     88,
         22380,    922,    459,   9435,    445,  11237,  24490, 128009, 128006,
         78191, 128007,    271,  10445,   1550,    279,   9435,    445,  11237,
         24490,   1464,    709,    449,    813,  23601,   1980,  18433,    568,
         15393,    568,    574,    304,    264,  21503,   5133,    449,    279,
           828,     11,    323,   1475,    892,    568,   6818,    311,   1646,
          1077,  21958,     11,   1364,   1120,   8434,    956,  80867,    311,
           264,  15528,   6425,     13, 128009], device='cuda:0')


In [19]:
new_tokenizer.decode(outputs[0])

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 30 Aug 2026\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nTell us a snarky joke about an applied LLM engineer<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nWhy did the applied LLM engineer break up with his girlfriend?\n\nBecause he realized he was in a toxic relationship with the data, and every time he tried to model her emotions, she just wouldn't converge to a stable solution.<|eot_id|>"

In [20]:
del model, new_tokenizer, inputs, outputs
gc.collect()
torch.cuda.empty_cache()

In [21]:
# # let's now write the bigger function, with all the details end to end, and which when passed arguments, returns the response
# # we pass in model, messages, whether to quant or not, max_new_tokens
# # now, we have to first create a tokenizer
def generate(model, messages, quant, max_new_tokens):
  gen_tokenizer = AutoTokenizer.from_pretrained(model)
  gen_tokenizer.pad_token = gen_tokenizer.eos_token
  gen_inputs = gen_tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
  if quant:
    gen_model = AutoModelForCausalLM.from_pretrained(model, quantization_config=quant_config, device_map='auto')
  else:
    gen_model = AutoModelForCausalLM.from_pretrained(model, device_map='auto')

  outputs = gen_model.generate(**gen_inputs, max_new_tokens=max_new_tokens)
  return gen_tokenizer.decode(outputs[0])

In [22]:
PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

messages = [
    {"role": "user", "content": "explain color blue to someone who can not see"}
]

generate(model=QWEN, messages=messages, quant=True, max_new_tokens=200)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

"<|im_start|>user\nexplain color blue to someone who can not see<|im_end|>\n<|im_start|>assistant\nThat's a great question — and a beautiful one, because it touches on how we understand the world beyond our senses.\n\nSince you can't see, you might not have direct experience of color, like blue. But we can still talk about *blue* in a meaningful, sensory and emotional way — using things you can feel, hear, think about, or even imagine.\n\nHere’s how we might explain **color blue** to someone who can't see:\n\n---\n\n### 🌫️ Blue: The Feeling of the Sky and the Sea\n\nImagine this:  \nWhen you look up at the sky on a clear day — especially in the morning or afternoon — it's often a soft, calm, open space. That feeling of peace, openness, and stillness? That’s part of what blue *feels like* in our minds.\n\nNow imagine the ocean — the vast, endless waves gently rolling in. The water might be calm, deep, and cool. That deep, cool feeling —"

In [25]:
generate(model=PHI, messages=messages, quant=True, max_new_tokens=200)

config.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json:   0%|          | 0.00/2.93k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 15.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

'<|user|>explain color blue to someone who can not see<|end|><|assistant|>Color blue is a visual perception that humans experience when looking at objects that reflect light in a certain way. Blue is one of the primary colors in the visible spectrum, which means it is a fundamental color that can be combined with other primary colors (red, green, and yellow) to create a wide range of other colors.\n\nTo someone who cannot see, the concept of color blue would be difficult to convey because it relies on visual perception. However, we can describe blue using other sensory experiences or analogies. For example, we might compare blue to the feeling of cool water or the sound of a gentle breeze. We could also describe blue as the color often associated with calmness, tranquility, and the sky on a clear day.\n\nWhile these descriptions can help convey the essence of blue, they cannot fully replicate the experience of seeing blue. For someone who is blind or visually impaired, other senses lik